In [1]:
!pip install yfinance pandas numpy scipy matplotlib seaborn dtaidistance scikit-learn statsmodels

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score, calinski_harabasz_score,davies_bouldin_score
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.stattools import adfuller, coint
from dtaidistance import dtw
import warnings
warnings.filterwarnings('ignore')

In [3]:
# --- Configuration ---
BENCHMARK = '^IXIC'
RISK_FREE_RATE = 0.045
START_DATE = '2014-01-01'
END_DATE = '2024-01-01'
ROLLING_WINDOW = 52
N_CLUSTERS = [2, 4, 7]  # Target number of clusters
MIN_HISTORY = 400 # Minimum weeks of data required per stock

In [4]:
# Retrieving NASDAQ tickers (100)
nasdaq_100_tickers = [
    'AAPL', 'MSFT', 'NVDA', 'AMZN', 'META', 'TSLA', 'GOOGL', 'GOOG', 'AVGO', 'ASML',
    'COST', 'NFLX', 'AZN', 'AMD', 'PEP', 'ADBE', 'LIN', 'QCOM', 'INTU', 'AMAT',
    'CSCO', 'TXN', 'AMGN', 'ISRG', 'MU', 'HON', 'BKNG', 'VRTX', 'REGN', 'ADI',
    'PANW', 'LRCX', 'GILD', 'SBUX', 'MDLZ', 'ADP', 'KLAC', 'MELI', 'SNPS', 'CDNS',
    'CRWD', 'ABNB', 'MAR', 'ORLY', 'CTAS', 'MNST', 'FTNT', 'WDAY', 'MRVL', 'PCAR',
    'CPRT', 'ROST', 'PAYX', 'DXCM', 'KDP', 'ODFL', 'FAST', 'IDXX', 'MCHP', 'BIIB',
    'EA', 'GEHC', 'EXC', 'KHC', 'VRSK', 'ON', 'XEL', 'CTSH', 'DLTR', 'CCEP',
    'CSGP', 'ANSS', 'FANG', 'BKR', 'CDW', 'TEAM', 'DDOG', 'SMCI', 'ILMN', 'WBD',
    'ZS', 'ALGN', 'ENPH', 'GFS', 'MDB', 'LCID', 'RIVN', 'SIRI', 'MTCH', 'PARA'
]

# Downloading benchmark data
print('Downloading NASDAQ benchmark data...')
benchmark_raw = yf.download(BENCHMARK, start=START_DATE, end=END_DATE, interval='1wk')
benchmark_prices = benchmark_raw['Close'].squeeze()
benchmark_returns = np.log(benchmark_prices / benchmark_prices.shift(1)).dropna()

print(f'Benchmark data downloaded: {len(benchmark_returns)} weekly observations')
print(f'Date range: {benchmark_returns.index[0].date()} to {benchmark_returns.index[-1].date()}')

[*********************100%***********************]  1 of 1 completed

Benchmark data downloaded: 521 weekly observations
Date range: 2014-01-08 to 2023-12-27


In [6]:
# Downloading all stock price data
print(f'Downloading price data for {len(nasdaq_100_tickers)} stocks...')

raw_data = yf.download(
    nasdaq_100_tickers,
    start=START_DATE,
    end=END_DATE,
    interval='1wk',
    auto_adjust=True,
    progress=True
)

# Extracting closing prices
price_data = raw_data['Close']

# Dropping stocks with insufficient history
valid_stocks = price_data.columns[price_data.count() >= MIN_HISTORY].tolist()
price_data = price_data[valid_stocks]

# Fill any small gaps, then drop any remaining NaNs
price_data = price_data.ffill().dropna(axis=1)

# Resampling to weekly frequency anchored on Wednesdays
price_data = price_data.resample('W-WED').last()
price_data = price_data.dropna(how='all')
valid_stocks = price_data.columns[price_data.count() >= MIN_HISTORY].tolist()
price_data = price_data[valid_stocks]
price_data = price_data.ffill().dropna(axis=1)


print(f'\nStocks with sufficient history: {len(price_data.columns)}')
print(f'Date range: {price_data.index[0].date()} to {price_data.index[-1].date()}')
print(f'Total weekly observations per stocks: {len(price_data)}')

print(price_data.shape)

[**                     4%                       ]  4 of 90 completed$PARA: possibly delisted; no timezone found
$ANSS: possibly delisted; no timezone found      ]  67 of 90 completed
[*********************100%***********************]  90 of 90 completed

2 Failed downloads:
['PARA', 'ANSS']: possibly delisted; no timezone found



Stocks with sufficient history: 77
Date range: 2014-01-01 to 2023-12-27
Total weekly observations per stocks: 522
(522, 77)


In [7]:
# Calculate Log Returns for all stocks and benchmarks
stock_returns = np.log(price_data / price_data.shift(1)).dropna()

# Align benchmark returns to match stock return dates exactly 
benchmark_returns_aligned = benchmark_returns.reindex(stock_returns.index).ffill().dropna()

# Making both the same date range
common_dates = stock_returns.index.intersection(benchmark_returns_aligned.index)
stock_returns = stock_returns.loc[common_dates]
benchmark_returns_aligned = benchmark_returns_aligned.loc[common_dates]

print(f'Stock returns shape: {stock_returns.shape}')
print(f'Benchmark returns shape: {benchmark_returns_aligned.shape}')
print(f'Date range: {stock_returns.index[0].date()} to {stock_returns.index[-1].date()}')
print(f'\nSample returns for AAPL (first 5 weeks):')
print(stock_returns['AAPL'].head())

Stock returns shape: (521, 77)
Benchmark returns shape: (521,)
Date range: 2014-01-08 to 2023-12-27

Sample returns for AAPL (first 5 weeks):
Date
2014-01-08    0.011690
2014-01-15    0.004893
2014-01-22   -0.080702
2014-01-29    0.004511
2014-02-05    0.052024
Freq: W-WED, Name: AAPL, dtype: float64


In [10]:
# Calculating Rolling Sharpe Ratio for each stock
def calculate_rolling_sharpe(stock_returns, risk_free_rate=RISK_FREE_RATE, window=ROLLING_WINDOW):
    """
    Sharpe Ratio = (Annualized Return - Risk Free Rate) / Annualized Volatility
    """

    # Converting annual risk free rate to weekly
    weekly_rf = risk_free_rate / 52

    # Calculating excess returns over risk free rate
    excess_returns = stock_returns.subtract(weekly_rf)

    # Calculating rolling annulaized mean excess return (Sharpe Ratio - numerator)
    rolling_mean = excess_returns.rolling(window=window).mean() * window 

    # Calculating rolling annualized volatilty (Sharpe Ratio - denominator)
    rolling_vol = stock_returns.rolling(window=window).std() * np.sqrt(window)

    # Sharpe Ratio
    sharpe = rolling_mean / rolling_vol

    return sharpe

# Running the Calculation
print("Calculating rolling Sharpe Ratios...")
sharpe_data = calculate_rolling_sharpe(stock_returns)

# Dropping the first 52 rows where rolling windows is not full yet
sharpe_data = sharpe_data.dropna(how='all')

# Dropping any stocks that still have NaN values
sharpe_data = sharpe_data.dropna(axis=1)

print(f"Sharpe Ratio matrix shape: {sharpe_data.shape}")
print(f"Date range: {sharpe_data.index[0].date()} to {sharpe_data.index[-1].date()}")
print(f"Stocks remaining: {len(sharpe_data.columns)}")
print(f"\nSample Sharpe Ratio for AAPL (first 5 values):")
print(sharpe_data['AAPL'].head())

Calculating rolling Sharpe Ratios...
Sharpe Ratio matrix shape: (470, 77)
Date range: 2014-12-31 to 2023-12-27
Stocks remaining: 77

Sample Sharpe Ratio for AAPL (first 5 values):
Date
2014-12-31    1.206897
2015-01-07    1.298953
2015-01-14    1.219418
2015-01-21    1.671935
2015-01-28    1.915245
Freq: W-WED, Name: AAPL, dtype: float64


In [12]:
# Building DTW Distance Matrix
print('Building DTW distance matrix...')
print('This is the most computationally expensive step, so it may take 5-10 minutes...')
n = len(sharpe_data.columns)
print(f'Computing {n} x {n} distance matrix\n')

# Convert Sharpe data to Numpy array
sharpe_matrix = sharpe_data.values.T # Transpose so shape is (n_stocks, n_timepoints)

# Normalizing each stock's IR series to have a mean of 0 and std of 1
scaler = StandardScaler()
sharpe_matrix_scaled = scaler.fit_transform(sharpe_matrix.T).T

# Computing DTW distance matrix
n_stocks = sharpe_matrix_scaled.shape[0]
distance_matrix_sharpe = np.zeros((n_stocks, n_stocks))

for i in range(n_stocks):
    for j in range(i+1, n_stocks):
        dist = dtw.distance(sharpe_matrix_scaled[i], sharpe_matrix_scaled[j])
        distance_matrix_sharpe[i, j] = dist
        distance_matrix_sharpe[j, i] = dist

    if i % 10 == 0:
        print(f'Progress: {i}/{n_stocks} stocks processed...')

print(f'\nDTW distance matrix complete!')
print(f'Distance matrix shape: {distance_matrix_sharpe.shape}')
print(f'Sample distances (first 3x3):')
print(np.round(distance_matrix[:3,:3], 4))

# Saving
np.save('distance_matrix_sharpe.npy', distance_matrix_sharpe)
print("\nDistance matrix saved to distance_matrix_sharpe.npy")

Building DTW distance matrix...
This is the most computationally expensive step, so it may take 5-10 minutes...
Computing 77 x 77 distance matrix

Progress: 0/77 stocks processed...
Progress: 10/77 stocks processed...
Progress: 20/77 stocks processed...
Progress: 30/77 stocks processed...
Progress: 40/77 stocks processed...
Progress: 50/77 stocks processed...
Progress: 60/77 stocks processed...
Progress: 70/77 stocks processed...

DTW distance matrix complete!
Distance matrix shape: (77, 77)
Sample distances (first 3x3):
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

Distance matrix saved to distance_matrix_sharpe.npy


In [ ]:
print("Sharpe matrix scaled shape:", sharpe_matrix_scaled.shape)
print("Any NaN values:", np.isnan(sharpe_matrix_scaled).any())
print("Any zero rows:", (sharpe_matrix_scaled == 0).all(axis=1).any())
print("Sample values (first stock, first 5 timepoints):")
print(sharpe_matrix_scaled[0, :5])
print("\nDistance matrix non-zero count:", np.count_nonzero(distance_matrix_sharpe))
print("Distance matrix max value:", distance_matrix_sharpe.max())
